# 🛠️ Notebook 2: Car Rental — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/car-rental
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import date, timedelta
from enum import Enum
from itertools import count

# ── Vehicles ──────────────────────────────────────────────
class Vehicle(ABC):
    def __init__(self, plate: str):
        self.plate = plate
    @abstractmethod
    def daily_rate(self) -> float: ...
    @abstractmethod
    def seats(self) -> int: ...
    def __repr__(self): return f"{type(self).__name__}({self.plate})"

class Car(Vehicle):
    def daily_rate(self): return 40
    def seats(self): return 5

class SUV(Vehicle):
    def daily_rate(self): return 65
    def seats(self): return 7

class Van(Vehicle):
    def daily_rate(self): return 80
    def seats(self): return 9

class Truck(Vehicle):
    def daily_rate(self): return 100
    def seats(self): return 3


In [ ]:
@dataclass
class Customer:
    id: int
    name: str
    license_no: str

class ReservationState(Enum):
    PENDING = "pending"; CONFIRMED = "confirmed"
    ACTIVE = "active"; RETURNED = "returned"; CANCELLED = "cancelled"

_ids = count(1)

@dataclass
class Reservation:
    customer: Customer
    vehicle: Vehicle
    start: date
    end: date
    id: int = field(default_factory=lambda: next(_ids))
    state: ReservationState = ReservationState.PENDING

    def overlaps(self, s: date, e: date) -> bool:
        return not (e < self.start or s > self.end)

    def days(self) -> int:
        return (self.end - self.start).days + 1

    def total(self) -> float:
        return self.days() * self.vehicle.daily_rate()

    def confirm(self):  self._require(ReservationState.PENDING);   self.state = ReservationState.CONFIRMED
    def pick_up(self):  self._require(ReservationState.CONFIRMED); self.state = ReservationState.ACTIVE
    def drop_off(self): self._require(ReservationState.ACTIVE);    self.state = ReservationState.RETURNED
    def cancel(self):
        if self.state in (ReservationState.RETURNED, ReservationState.ACTIVE):
            raise ValueError("cannot cancel a picked-up reservation")
        self.state = ReservationState.CANCELLED

    def _require(self, s):
        if self.state != s: raise ValueError(f"expected {s}, got {self.state}")


In [ ]:
class RentalStore:
    def __init__(self, fleet: list[Vehicle]):
        self.fleet = fleet
        self.reservations: list[Reservation] = []

    def available(self, start: date, end: date) -> list[Vehicle]:
        busy = {r.vehicle.plate for r in self.reservations
                if r.state in (ReservationState.CONFIRMED, ReservationState.ACTIVE)
                and r.overlaps(start, end)}
        return [v for v in self.fleet if v.plate not in busy]

    def book(self, customer: Customer, vehicle: Vehicle, start: date, end: date) -> Reservation:
        if vehicle not in self.available(start, end):
            raise ValueError("vehicle not available on those dates")
        r = Reservation(customer, vehicle, start, end)
        r.confirm()
        self.reservations.append(r)
        return r


store = RentalStore([Car("ABC-1"), Car("ABC-2"), SUV("XYZ-9"), Van("VAN-7")])
alice = Customer(1, "Alice", "DL-111")
bob   = Customer(2, "Bob",   "DL-222")

today = date(2025, 1, 1)
r1 = store.book(alice, store.fleet[0], today, today + timedelta(days=2))
print(r1, "→ total $", r1.total())

# Overlapping booking of the same car should fail
try:
    store.book(bob, store.fleet[0], today + timedelta(days=1), today + timedelta(days=3))
except ValueError as e:
    print("expected:", e)

# But a different car is fine
r2 = store.book(bob, store.fleet[1], today + timedelta(days=1), today + timedelta(days=3))
print(r2, "→ total $", r2.total())

r1.pick_up(); r1.drop_off()
print("r1 final state:", r1.state)


### Try it
- Add a `Payment` class and charge on `pick_up`.
- Add late-return fees (a day rate + penalty) in `drop_off(actual_date)`.
- Extend `available()` with a `category` filter (e.g., only SUVs).